In [ ]:
# ============================================================
# TIER 2: Mango Leaf Damage + Severity Estimator
# ============================================================

import cv2
import numpy as np
import pandas as pd
from pathlib import Path

# -------- SETTINGS --------
image_dir = "../../Database/dataset/Anthracnose/" 
output_csv = "leaf_features_tier2.csv"
cm_per_pixel = 0.01   # calibrate using a known reference

# -------- UTILITIES --------
def area_cm2(pixels, cm_per_pixel):
    return pixels * (cm_per_pixel ** 2)

def segment_leaf(img_hsv):
    lower = np.array([20, 40, 40])
    upper = np.array([90, 255, 255])
    mask = cv2.inRange(img_hsv, lower, upper)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, np.ones((5,5),np.uint8))
    return mask

def segment_lesion(img_hsv, leaf_mask):
    lower = np.array([5, 30, 0])
    upper = np.array([25, 255, 160])
    mask = cv2.inRange(img_hsv, lower, upper)
    mask = cv2.bitwise_and(mask, leaf_mask)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, np.ones((3,3),np.uint8))
    return mask

# -------- MAIN LOOP --------
records = []

for path in Path(image_dir).glob("*.jpg"):
    img = cv2.imread(str(path))
    if img is None:
        continue

    img = cv2.resize(img, (800, int(img.shape[0]*800/img.shape[1])))
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)

    # --- segmentation ---
    leaf_mask = segment_leaf(hsv)
    lesion_mask = segment_lesion(hsv, leaf_mask)
    contours, _ = cv2.findContours(leaf_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        continue
    largest_contour = max(contours, key=cv2.contourArea)

    # --- base areas ---
    leaf_area_px = cv2.countNonZero(leaf_mask)
    lesion_area_px = cv2.countNonZero(lesion_mask)
    leaf_area_cm2 = area_cm2(leaf_area_px, cm_per_pixel)
    lesion_area_cm2 = area_cm2(lesion_area_px, cm_per_pixel)

    severity_pct = (lesion_area_px / leaf_area_px * 100) if leaf_area_px > 0 else 0

    # --- reconstruction: convex hull ---
    hull = cv2.convexHull(largest_contour)
    hull_area = cv2.contourArea(hull)
    hull_area_cm2 = area_cm2(hull_area, cm_per_pixel)
    missing_pct_hull = max((hull_area - leaf_area_px) / hull_area * 100, 0)

    # --- reconstruction: ellipse fit (if possible) ---
    ellipse_area_px = None
    missing_pct_ellipse = None
    if len(largest_contour) >= 5:
        ellipse = cv2.fitEllipse(largest_contour)
        ellipse_area_px = np.pi * (ellipse[1][0]/2) * (ellipse[1][1]/2)
        missing_pct_ellipse = max((ellipse_area_px - leaf_area_px) / ellipse_area_px * 100, 0)

    # --- color stats ---
    leaf_mean = cv2.mean(img, mask=leaf_mask)[:3]
    lesion_mean = cv2.mean(img, mask=lesion_mask)[:3]

    # --- save record ---
    records.append({
        "filename": path.name,
        "leaf_area_px": leaf_area_px,
        "lesion_area_px": lesion_area_px,
        "leaf_area_cm2": leaf_area_cm2,
        "lesion_area_cm2": lesion_area_cm2,
        "severity_pct": severity_pct,
        "hull_area_px": hull_area,
        "hull_missing_pct": missing_pct_hull,
        "ellipse_area_px": ellipse_area_px,
        "ellipse_missing_pct": missing_pct_ellipse,
        "leaf_mean_B": leaf_mean[0],
        "leaf_mean_G": leaf_mean[1],
        "leaf_mean_R": leaf_mean[2],
        "lesion_mean_B": lesion_mean[0],
        "lesion_mean_G": lesion_mean[1],
        "lesion_mean_R": lesion_mean[2]
    })

    # --- visualization ---
    overlay = img.copy()
    overlay[leaf_mask == 0] = (0, 0, 0)
    cv2.drawContours(overlay, [hull], -1, (255, 0, 0), 2)  # blue hull
    cv2.drawContours(overlay, cv2.findContours(lesion_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)[0], -1, (0, 0, 255), 2)
    cv2.putText(overlay, f"Severity: {severity_pct:.1f}%  Missing: {missing_pct_hull:.1f}%", (20,40),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255,255,255), 2)
    cv2.imshow("Leaf Analysis", overlay)
    cv2.waitKey(200)

cv2.destroyAllWindows()

# -------- SAVE --------
df = pd.DataFrame(records)
df.to_csv(output_csv, index=False)
print("✅ Tier 2 extraction complete.")
print(df.head())
# ============================================================

✅ Tier 2 extraction complete.
                       filename  leaf_area_px  lesion_area_px  leaf_area_cm2  \
0  20211008_124249 (Custom).jpg        188483           76080        18.8483   
1  20211008_124250 (Custom).jpg        176272           66566        17.6272   
2  20211008_124252 (Custom).jpg        198491           70074        19.8491   
3  20211008_124253 (Custom).jpg        175075           60201        17.5075   
4  20211008_124256 (Custom).jpg        227100          119754        22.7100   

   lesion_area_cm2  severity_pct  hull_area_px  hull_missing_pct  \
0           7.6080     40.364383      199510.0          5.527041   
1           6.6566     37.763230      186933.0          5.703113   
2           7.0074     35.303364      210189.5          5.565692   
3           6.0201     34.385835      188495.0          7.119552   
4          11.9754     52.731836      248839.5          8.736354   

   ellipse_area_px  ellipse_missing_pct  leaf_mean_B  leaf_mean_G  \
0    427116

In [7]:
# ============================================================
# TIER 2: Mango Leaf Damage + Severity Estimator (with Inpainting)
# ============================================================

import cv2
import numpy as np
import pandas as pd
from pathlib import Path

# -------- SETTINGS --------
image_dir = "../../Database/dataset/Anthracnose/" 
output_csv = "leaf_features_tier2.csv"
cm_per_pixel = 0.01   # calibrate using a known reference

# -------- UTILITIES --------
def area_cm2(pixels, cm_per_pixel):
    return pixels * (cm_per_pixel ** 2)

def segment_leaf(img_hsv):
    lower = np.array([20, 40, 40])
    upper = np.array([90, 255, 255])
    mask = cv2.inRange(img_hsv, lower, upper)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, np.ones((5,5), np.uint8))
    return mask

def segment_lesion(img_hsv, leaf_mask):
    lower = np.array([5, 30, 0])
    upper = np.array([25, 255, 160])
    mask = cv2.inRange(img_hsv, lower, upper)
    mask = cv2.bitwise_and(mask, leaf_mask)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, np.ones((3,3), np.uint8))
    return mask

def reconstruct_leaf_inpaint(img, leaf_mask):
    """Use OpenCV inpainting to reconstruct missing/damaged leaf areas."""
    # Find holes inside the leaf mask (invert)
    holes = cv2.bitwise_not(leaf_mask)
    holes = cv2.morphologyEx(holes, cv2.MORPH_OPEN, np.ones((5,5), np.uint8))
    
    # Create inpainting mask only inside convex hull of leaf
    contours, _ = cv2.findContours(leaf_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return leaf_mask, 0, img
    
    largest_contour = max(contours, key=cv2.contourArea)
    hull = cv2.convexHull(largest_contour)
    hull_mask = np.zeros_like(leaf_mask)
    cv2.drawContours(hull_mask, [hull], -1, 255, -1)
    inpaint_mask = cv2.bitwise_and(holes, hull_mask)

    # Estimate missing percentage (how much of hull is empty)
    missing_px = cv2.countNonZero(inpaint_mask)
    hull_area = cv2.countNonZero(hull_mask)
    damage_pct_inpaint = (missing_px / hull_area * 100) if hull_area > 0 else 0

    # Perform inpainting on original image for visualization
    inpainted_img = cv2.inpaint(img, inpaint_mask, 5, cv2.INPAINT_TELEA)

    # Reconstruct leaf mask (fill holes)
    reconstructed_mask = cv2.bitwise_or(leaf_mask, inpaint_mask)

    return reconstructed_mask, damage_pct_inpaint, inpainted_img

# -------- MAIN LOOP --------
records = []

for path in Path(image_dir).glob("*.jpg"):
    img = cv2.imread(str(path))
    if img is None:
        continue

    img = cv2.resize(img, (800, int(img.shape[0]*800/img.shape[1])))
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)

    # --- segmentation ---
    leaf_mask = segment_leaf(hsv)
    lesion_mask = segment_lesion(hsv, leaf_mask)

    # --- reconstruction (inpainting) ---
    leaf_mask_recon, damage_pct_inpaint, inpainted_img = reconstruct_leaf_inpaint(img, leaf_mask)

    contours, _ = cv2.findContours(leaf_mask_recon, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        continue
    largest_contour = max(contours, key=cv2.contourArea)

    # --- base areas ---
    leaf_area_px = cv2.countNonZero(leaf_mask_recon)
    lesion_area_px = cv2.countNonZero(lesion_mask)
    leaf_area_cm2 = area_cm2(leaf_area_px, cm_per_pixel)
    lesion_area_cm2 = area_cm2(lesion_area_px, cm_per_pixel)
    severity_pct = (lesion_area_px / leaf_area_px * 100) if leaf_area_px > 0 else 0

    # --- convex hull + ellipse reconstruction ---
    hull = cv2.convexHull(largest_contour)
    hull_area = cv2.contourArea(hull)
    hull_area_cm2 = area_cm2(hull_area, cm_per_pixel)
    missing_pct_hull = max((hull_area - leaf_area_px) / hull_area * 100, 0)

    ellipse_area_px = None
    missing_pct_ellipse = None
    if len(largest_contour) >= 5:
        ellipse = cv2.fitEllipse(largest_contour)
        ellipse_area_px = np.pi * (ellipse[1][0]/2) * (ellipse[1][1]/2)
        missing_pct_ellipse = max((ellipse_area_px - leaf_area_px) / ellipse_area_px * 100, 0)

    # --- color stats ---
    leaf_mean = cv2.mean(img, mask=leaf_mask_recon)[:3]
    lesion_mean = cv2.mean(img, mask=lesion_mask)[:3]

    # --- save record ---
    records.append({
        "filename": path.name,
        "leaf_area_px": leaf_area_px,
        "lesion_area_px": lesion_area_px,
        "leaf_area_cm2": leaf_area_cm2,
        "lesion_area_cm2": lesion_area_cm2,
        "severity_pct": severity_pct,
        "hull_area_px": hull_area,
        "hull_missing_pct": missing_pct_hull,
        "ellipse_area_px": ellipse_area_px,
        "ellipse_missing_pct": missing_pct_ellipse,
        "damage_pct_inpaint": damage_pct_inpaint,
        "leaf_mean_B": leaf_mean[0],
        "leaf_mean_G": leaf_mean[1],
        "leaf_mean_R": leaf_mean[2],
        "lesion_mean_B": lesion_mean[0],
        "lesion_mean_G": lesion_mean[1],
        "lesion_mean_R": lesion_mean[2]
    })

    # --- visualization ---
    overlay = inpainted_img.copy()
    overlay[leaf_mask_recon == 0] = (0, 0, 0)
    cv2.drawContours(overlay, [hull], -1, (255, 0, 0), 2)
    cv2.drawContours(overlay, cv2.findContours(lesion_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)[0], -1, (0, 0, 255), 2)
    cv2.putText(overlay, f"Severity: {severity_pct:.1f}% | Missing: {missing_pct_hull:.1f}% | Repaired: {damage_pct_inpaint:.1f}%", 
                (20,40), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255,255,255), 2)
    cv2.imshow("Leaf Analysis", overlay)
    cv2.waitKey(200)

cv2.destroyAllWindows()

# -------- SAVE --------
df = pd.DataFrame(records)
df.to_csv(output_csv, index=False)
print("✅ Tier 2 extraction (with inpainting) complete.")
print(df.head())
# ============================================================


✅ Tier 2 extraction (with inpainting) complete.
                       filename  leaf_area_px  lesion_area_px  leaf_area_cm2  \
0  20211008_124249 (Custom).jpg        200792           76080        20.0792   
1  20211008_124250 (Custom).jpg        188037           66566        18.8037   
2  20211008_124252 (Custom).jpg        211292           70074        21.1292   
3  20211008_124253 (Custom).jpg        189573           60201        18.9573   
4  20211008_124256 (Custom).jpg        250026          119754        25.0026   

   lesion_area_cm2  severity_pct  hull_area_px  hull_missing_pct  \
0           7.6080     37.889956      200208.5               0.0   
1           6.6566     35.400480      187702.0               0.0   
2           7.0074     33.164531      211133.5               0.0   
3           6.0201     31.756105      189406.0               0.0   
4          11.9754     47.896619      249767.0               0.0   

   ellipse_area_px  ellipse_missing_pct  damage_pct_inpaint  l